In [ ]:
"""
MAHE SDG Scraper — undetected-chromedriver (Chrome)
Run from Anaconda Prompt:
    python scraper_chrome.py

undetected-chromedriver patches Chrome binaries to bypass Cloudflare.
No manual driver download needed — it handles everything automatically.

Output: mahe_sdg_publications.csv
"""

import time
import pandas as pd
from bs4 import BeautifulSoup

import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# CONFIG 
TOTAL_PAGES      = 20
FETCH_ABSTRACTS  = True
HEADLESS         = False    # Keep False — undetected-chromedriver works best visible
CHECKPOINT_EVERY = 2
OUTPUT_CSV       = "mahe_sdg_publications.csv"

BASE_URL = (
    "https://researcher.manipal.edu/en/publications/"
    "?search=SDG&originalSearch=SDG&pageSize=50"
    "&ordering=rating&descending=true&showAdvanced=false"
    "&allConcepts=true&inferConcepts=true&searchBy=RelatedConcepts&page="
)

# Driver 
def init_driver():
    options = uc.ChromeOptions()
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    driver = uc.Chrome(
        options=options,
        headless=HEADLESS,
        version_main=147,       # your Chrome major version
        use_subprocess=True,    # required to avoid asyncio conflicts
    )
    return driver

# Cloudflare check 
def is_cloudflare(driver):
    src = driver.page_source.lower()
    return (
        "security verification" in src or
        "cf-turnstile"          in src or
        "verify you are human"  in src or
        "checking your browser" in src or
        "enable javascript"     in src
    )

# Abstract fetcher 
def fetch_abstract(driver, url):
    try:
        driver.get(url)
        time.sleep(5)

        # If Cloudflare reappears on article page, wait it out
        if is_cloudflare(driver):
            time.sleep(15)

        soup = BeautifulSoup(driver.page_source, "html.parser")
        for sel in [
            "div.rendering_abstractportal",
            "div.textblock",
            "div.rendering_researchoutput_short",
            "section.abstract",
            "div[class*='abstract']",
            "p.abstract",
        ]:
            el = soup.select_one(sel)
            if el:
                return el.get_text(separator=" ", strip=True)
    except Exception as e:
        print(f"    ⚠ Abstract error: {e}")
    return ""

# Main 
def main():
    all_data  = []
    seen_urls = set()

    print("=" * 55)
    print("  MAHE SDG Scraper — Chrome (undetected)")
    print("=" * 55)
    print("Starting Chrome...")

    driver = init_driver()

    try:
        # Page 1 — handle Cloudflare 
        print("\nLoading page 1...")
        driver.get(BASE_URL + "1")
        time.sleep(8)   # give undetected-chromedriver time to auto-solve

        # Check if Cloudflare is still showing
        if is_cloudflare(driver):
            print("\n" + "=" * 55)
            print("  Cloudflare detected.")
            print("  undetected-chromedriver will attempt to auto-solve.")
            print("  If a challenge appears in the browser,")
            print("  click 'Verify you are human' manually.")
            print("  Waiting up to 40 seconds...")
            print("=" * 55)

            for i in range(8):
                time.sleep(5)
                if not is_cloudflare(driver):
                    print("  ✅ Cloudflare cleared automatically!")
                    break
                print(f"  ⏳ Still waiting... ({(i+1)*5}/40s)")
            else:
                if is_cloudflare(driver):
                    print("  ❌ Could not bypass Cloudflare. Exiting.")
                    driver.quit()
                    return

        print("  ✅ Portal accessible — starting scrape...")

        # Scrape all pages 
        for page_num in range(1, TOTAL_PAGES + 1):
            print(f"\n─── Page {page_num}/{TOTAL_PAGES} ──────────────────────")

            try:
                if page_num > 1:
                    driver.get(BASE_URL + str(page_num))
                    time.sleep(6)

                # Cloudflare recheck
                if is_cloudflare(driver):
                    print("  ⚠ Cloudflare reappeared — waiting 30s...")
                    time.sleep(30)
                    if is_cloudflare(driver):
                        print(f"  ❌ Skipping page {page_num}")
                        continue

                # Scroll to trigger lazy loading
                for _ in range(3):
                    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                    time.sleep(2)

                # Wait for articles to appear
                try:
                    WebDriverWait(driver, 15).until(
                        EC.presence_of_all_elements_located(
                            (By.CSS_SELECTOR, "h3 > a[href]")
                        )
                    )
                except:
                    pass

                soup    = BeautifulSoup(driver.page_source, "html.parser")
                entries = soup.select("h3 > a[href]")

                # Fallback selectors
                if not entries:
                    entries = soup.select(
                        "a.title, a[class*='title'], h2 > a, h4 > a"
                    )

                print(f"  Found {len(entries)} articles")

                if len(entries) == 0:
                    print("  ⚠ No articles found — printing page snippet for debug:")
                    print(driver.page_source[:1000])
                    continue

                for entry in entries:
                    title    = entry.get_text(strip=True)
                    rel_link = entry.get("href", "")
                    full_url = (
                        f"https://researcher.manipal.edu{rel_link}"
                        if rel_link.startswith("/") else rel_link
                    )

                    if full_url in seen_urls or not title:
                        continue
                    seen_urls.add(full_url)

                    parent = entry.find_parent("div")
                    tags   = [
                        t.get_text(strip=True)
                        for t in parent.select("ul.relations li span.label")
                    ] if parent else []

                    year = ""
                    if parent:
                        yr_el = parent.select_one(
                            "span.date, time, span[class*='year']"
                        )
                        if yr_el:
                            year = yr_el.get_text(strip=True)

                    abstract = ""
                    if FETCH_ABSTRACTS and full_url:
                        abstract = fetch_abstract(driver, full_url)
                        driver.back()
                        time.sleep(3)

                    all_data.append({
                        "Title":       title,
                        "URL":         full_url,
                        "Year":        year,
                        "Abstract":    abstract[:800],
                        "Portal Tags": ", ".join(tags),
                    })

                print(f"  Total collected: {len(all_data)}")

                # Checkpoint save
                if page_num % CHECKPOINT_EVERY == 0:
                    pd.DataFrame(all_data).to_csv(
                        f"checkpoint_page{page_num}.csv", index=False
                    )
                    print(f"  💾 Checkpoint saved")

            except Exception as e:
                print(f"  ❌ Page {page_num} error: {e}")
                continue

    finally:
        driver.quit()
        print("\n🔒 Browser closed")

    # Save 
    if not all_data:
        print("\n❌ No data collected.")
        return

    df = pd.DataFrame(all_data)
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    print(f"\n{'='*55}")
    print(f"  SCRAPING COMPLETE")
    print(f"  Total articles   : {len(df)}")
    print(f"  With abstract    : {(df['Abstract'].str.len() > 10).sum()}")
    print(f"  Without abstract : {(df['Abstract'].str.len() <= 10).sum()}")
    print(f"  Saved to         : {OUTPUT_CSV}")
    print(f"{'='*55}")
    print(df[["Title", "Year", "Abstract"]].head(5).to_string())


if __name__ == "__main__":
    main()
